# SPACE Headless API Example

Demonstrates the `space_api.run_pipeline()` headless API for the SPACE
(Sequence Protein Alignment and Conservation Engine) package.

Interpreter: `/home/dzyla/miniconda3/envs/space/bin/python`


In [1]:
# Ensure the SPACE repo root is on sys.path (works from any launch dir)
import sys, os
def _find_space_root():
    # look for space_api.py starting from this notebook dir upward
    d = os.path.abspath(os.getcwd())
    for _ in range(6):
        if os.path.isfile(os.path.join(d, "space_api.py")):
            return d
        d = os.path.dirname(d)
    return os.getcwd()
root = _find_space_root()
if root not in sys.path:
    sys.path.insert(0, root)
import space_api as sa


## Run the full pipeline headlessly

`space_api.run_pipeline()` installs the headless Streamlit stub internally
(`headless_streamlit.install()`), so the entire `space.*` pipeline runs without
a browser. The NCBI email is `dzyla@lji.org`.


In [2]:
result = sa.run_pipeline(
    query="Hendra henipavirus F",
    email="dzyla@lji.org",
    data_source="UniProt",
    max_seqs=50,
    out_dir="runs/hendra_f",
)

print("Pipeline completed. Output dir:", result["out_dir"])
print("FASTA:", result["fasta_path"])
print("MSA:", result["msa_outfile"])
print("ALN:", result["aln_file"])
print("Tree:", result["tree_file"])


Pipeline completed. Output dir: runs/hendra_f
FASTA: runs/hendra_f/sequences.fasta
MSA: runs/hendra_f/msa/msa_out.fasta
ALN: runs/hendra_f/msa/msa_out.aln
Tree: runs/hendra_f/tree/phylogenetic_tree.nwk


## Inspect the outputs

The returned dict contains: `al2co_df`, `conservation_df`, `mutations_df`,
`tree_file`, and optionally `pdb_result` (when `map_to_structure=True`).


In [3]:
# al2co conservation scores (per-residue)
print("=== al2co_df ===")
print(result["al2co_df"].head())

# point mutations
print("\n=== mutations_df ===")
print(result["mutations_df"].head())

# conservation summary
print("\n=== conservation_df ===")
print(result["conservation_df"].head())


=== al2co_df ===
   Location Residue  al2co_score
0       1.0       -         -1.0
1       2.0       -         -1.0
2       3.0       -         -1.0
3       4.0       -         -1.0
4       5.0       -         -1.0

=== mutations_df ===
              Sequence ID Original Residue  Position Mutated Residue  \
0  tr_G5CPV3_G5CPV3_9PARA                G       325               E   
1  tr_G5CPV3_G5CPV3_9PARA                V       398               I   
2  tr_G5CPV3_G5CPV3_9PARA                R       412               G   
3  tr_G5CPV3_G5CPV3_9PARA                M       645               T   
4  tr_G5CPV3_G5CPV3_9PARA                Q       676               K   

  Mutation Type Mutation  
0  Substitution    G325E  
1  Substitution    V398I  
2  Substitution    R412G  
3  Substitution    M645T  
4  Substitution    Q676K  

=== conservation_df ===
   Location  Conservation
0       1.0      0.151515
1       2.0      0.151515
2       3.0      0.151515
3       4.0      0.151515
4       5.0  

## Optional structural mapping (Step 7)

Map al2co scores onto a 3D structure by setting `map_to_structure=True` and
providing a `uniprot_id` (fetches the AlphaFold PDB) or `own_pdb`.


In [4]:
result_struct = sa.run_pipeline(
    query="Hendra henipavirus F",
    email="dzyla@lji.org",
    data_source="UniProt",
    max_seqs=50,
    out_dir="runs/hendra_f_struct",
    map_to_structure=True,
    uniprot_id="Q8IHW4",
)

print("Structural mapping result:", result_struct["pdb_result"])


Structural mapping result: {'metadata': {'pdb_id': 'Q8IHW4 (AlphaFold)', 'title': 'V-type proton ATPase subunit F', 'authors': 'AlphaFold DB', 'date': 'N/A'}, 'selected_pdb': 'Q8IHW4', 'chain_data': {'A': {'alignment_score': -400.6000000000144, 'matched_length': 127, 'alignment': 'target            0 MAHELSISDIIYPECHLDSPIVSGKLISAIEYAQLRHNQPNGDKRLTENIKINLQGKRRS\n                  0 ||----------------------------------------------------------\nquery             0 MA----------------------------------------------------------\n\ntarget           60 VYISRQSRLGNYIRDNIKNLKEFLHVSYPECNKSLFSLKSPGMTSKLSNIMKKSFKAYNI\n                 60 ------||-----|----------|-----------------------------------\nquery             2 ------SR-----R----------H-----------------------------------\n\ntarget          120 VSRKIIEMLQNITRNLITQDQKDEVLGIYEQDRLSNIGKYMSQSQWYECFLFWFTIKTEM\n                120 --------------------------------||--------------------------\nquery             6 --------------------------------RL----

## Pitfall: identical sequences yield zero variance

If all aligned sequences are identical, every column has zero variance and
al2co returns NaN conservation scores. Deduplicate identical sequences before
running the pipeline.


In [5]:
import tempfile, os
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord

identical = [
    SeqRecord(Seq("ACDEFGHIKLMNPQRSTVWY"), id="seq1"),
    SeqRecord(Seq("ACDEFGHIKLMNPQRSTVWY"), id="seq2"),
    SeqRecord(Seq("ACDEFGHIKLMNPQRSTVWY"), id="seq3"),
]
with tempfile.TemporaryDirectory() as tmpdir:
    fasta_path = os.path.join(tmpdir, "identical.fasta")
    SeqIO.write(identical, fasta_path, "fasta")
    try:
        result_id = sa.run_pipeline(
            fasta_path=fasta_path,
            out_dir=os.path.join(tmpdir, "identical_run"),
        )
        print("al2co scores for identical sequences:")
        print(result_id["al2co_df"].head())
        print("Any NaN?", result_id["al2co_df"].isna().any().any())
    except FileNotFoundError as e:
        print("EXPECTED PITFALL — identical sequences give al2co zero variance,")
        print("so the native al2co binary writes no output and raises:")
        print("  ", e)
        print("Fix: deduplicate identical sequences before running the pipeline.")


EXPECTED PITFALL — identical sequences give al2co zero variance,
so the native al2co binary writes no output and raises:
   [Errno 2] No such file or directory: '/tmp/tmpnpcjiy73.aln.out'
Fix: deduplicate identical sequences before running the pipeline.


## Pipeline stage reference

| Stage | Function | Output |
|---|---|---|
| Step 0 - fetch | `fetch_sequences` | `sequences.fasta` |
| Step 1 - pairwise align | `perform_alignment` | identity/coverage, mapping |
| Step 2 - filter | `filter_sequences` | filtered sequences |
| Step 3 - MSA | `perform_msa_pyfamsa` | `msa_out.fasta`, `.aln` |
| Step 4 - al2co | `run_al2co` | `al2co_df` |
| Step 5 - mutations | `parse_mutations` | `mutations_df` |
| Step 6 - tree | `generate_phylogenetic_tree` | tree file (if <=200 seqs) |
| Step 7 - structure | `map_structure` | `pdb_result` (optional) |
